[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_05/MFM_ch5_seminar/MFM_ch5_seminar.ipynb)

# Modelling Financial Markets — Seminar 5: Conditional Volatility, GARCH Models

*Seminar notebook. Daniel Traian Pele, Antoaneta Amza, Bucharest University of Economic Studies.*

- **[Solved]** problems contain the full solution; **[Proposed]** problems are solved by you.
- Part A: derivations (A1–A6); Part B: computation and inference (B1–B8); Part C: open questions and AI critique (C1 US spillover to the BET, C2 critique of an AI answer, C3 quasi score-driven GARCH).

> Quantlet `MFM_ch5_seminar`: student version of the Seminar 5 notebook. Solved exercises (A1, A3, B1, B2, B5, B5 Extended, B7) are shown with code and output; the proposed exercises (A2, A4, A5, A6, B3, B4, B6, B8, C1, C2, C3) are not solved here (an empty code cell where they need code).

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_5'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# the arch package (Kevin Sheppard) is not preinstalled on Colab
try:
    import arch
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats, optimize
from scipy.signal import lfilter
from scipy.special import gammaln
from arch import arch_model
from arch.univariate import ARCHInMean, ConstantMean, GARCH, FIGARCH, StudentsT, ZeroMean
import warnings
warnings.filterwarnings('ignore')

## Chart style and output folder

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
# font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 11.5
plt.rcParams['ytick.labelsize'] = 11.5
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.4
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 11

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'btc': Amber, 'eurron': Forest, 'gold': Purple}

SEED = 42
TABLE_DIR = '.'
NUM = {}


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def acf(x, nlags):
    x = np.asarray(x) - np.mean(x)
    d = np.sum(x ** 2)
    return np.array([np.sum(x[k:] * x[:-k]) / d for k in range(1, nlags + 1)])


import tempfile
# outputs go to OUT_DIR (Google Drive) if SAVE_TO_DRIVE = True, otherwise to a temporary folder
OUT_PATH = OUT_DIR if globals().get('SAVE_TO_DRIVE') else tempfile.gettempdir()
TABLE_DIR = OUT_PATH


def save_fig(name):
    """Save the figure as PDF and PNG in OUT_PATH, then show it."""
    plt.savefig(os.path.join(OUT_PATH, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(OUT_PATH, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()

## Data

- Daily closes up to 18 September 2026; returns in %: $r_t = 100(\ln P_t - \ln P_{t-1})$.
- Equity indices and gold: weekdays only, days with an unchanged close (filled holidays) removed; gold is annualised with its actual frequency.
- Bitcoin trades 7 days a week; EUR/RON is the BNR official reference rate.
- Each series is analysed on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date)
MARKETS = {
    'sp500':  ('GSPC.INDX',      'S&P 500',                 'Equity', '2000-01-01'),
    'bet':    ('BET',            'BET',                     'Equity', '2000-01-01'),
    'bettr':  ('BETTR.INDX',     'BET-TR',                  'Equity', '2014-09-23'),
    'btc':    ('BTC-USD.CC',     'Bitcoin',                 'Crypto', '2014-09-17'),
    'eurron': ('REF:EUR',        'EUR/RON (BNR reference)', 'FX',     '2005-07-01'),
    'gold':   ('XAUUSD.FOREX',   'Gold (XAU/USD)',          'Commodity', '2000-01-01'),
}
LABELS = {k: v[1] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one symbol (local copy of the course data, or GitHub)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (annual BNR XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0 = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    return s.rename(name)


def pct_returns(name, start=None, end=END):
    """Daily log returns in percent, on the series' own calendar."""
    return (100 * np.log(load_close(name, start, end)).diff().dropna()).rename(name)


def periods_per_year(r):
    """Average number of observations per calendar year (actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def load_vix(start='2000-01-01', end=END):
    """The VIX index (30-day implied volatility of the S&P 500, annualised percent)."""
    s = read_market('VIX.INDX')['close'].loc[start:end]
    return s[s.index.dayofweek < 5].rename('vix')


rets = {k: pct_returns(k) for k in MARKETS}
PPY = {k: periods_per_year(rets[k]) for k in MARKETS}
ORDER = list(MARKETS)
MAIN = ['sp500', 'bet', 'btc', 'eurron', 'gold']
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs',
          round(PPY[k], 1), 'obs/year')

## GARCH tools

- `fit_garch`: GARCH, GJR-GARCH or EGARCH with Normal, Student-t, skewed-t or GED innovations (package `arch`); low-volatility series are rescaled internally and the parameters converted back.
- `half_life`, `news_impact`, `sign_bias_test`, `ljung_box`, `arch_lm`, `ewma_variance`, `qlike`, `mz_regression`, `dm_test`.

In [ ]:
VOL_SPEC = {'GARCH': dict(vol='GARCH', p=1, o=0, q=1),
            'GJR': dict(vol='GARCH', p=1, o=1, q=1),
            'EGARCH': dict(vol='EGARCH', p=1, o=1, q=1)}
DISTS = {'normal': 'Normal', 't': 'Student-t', 'skewt': 'Skewed-t', 'ged': 'GED'}


class Fit:
    """Result of one fit, in the units of the original series (returns in %)."""

    def __init__(self, res, c, r, vol):
        self.res, self.c, self.r, self.vol = res, c, r, vol
        p = res.params.copy()
        p['mu'] = p['mu'] / c
        if vol == 'EGARCH':
            p['omega'] = p['omega'] - (1 - p['beta[1]']) * np.log(c ** 2)
        else:
            p['omega'] = p['omega'] / c ** 2
        self.params = p
        # robust (Bollerslev-Wooldridge) covariance mapped to the original units with the Jacobian
        # of the transformation: mu/c, omega/c^2 (GARCH, GJR) or omega - (1 - beta) ln c^2 (EGARCH)
        names = list(res.params.index)
        J = np.eye(len(names))
        J[names.index('mu'), names.index('mu')] = 1 / c
        if vol == 'EGARCH':
            J[names.index('omega'), names.index('beta[1]')] = np.log(c ** 2)
        else:
            J[names.index('omega'), names.index('omega')] = 1 / c ** 2
        self.cov = pd.DataFrame(J @ res.param_cov.values @ J.T, index=names, columns=names)
        self.se = pd.Series(np.sqrt(np.diag(self.cov.values)), index=names)
        self.tvalues = p[names] / self.se
        self.pvalues = pd.Series(2 * stats.norm.sf(np.abs(self.tvalues.values)), index=names)
        n = res.nobs
        self.nobs = n
        self.loglik = res.loglikelihood + n * np.log(c)
        k = len(p)
        self.aic = -2 * self.loglik + 2 * k
        self.bic = -2 * self.loglik + k * np.log(n)
        self.sigma = res.conditional_volatility / c
        self.z = res.std_resid
        self.eps = res.resid / c                   # unstandardised residuals, in the units of the original series
        self.converged = res.convergence_flag == 0

    @property
    def persistence(self):
        p = self.params
        if self.vol == 'EGARCH':
            return p['beta[1]']
        g = p.get('gamma[1]', 0.0)
        return p['alpha[1]'] + p['beta[1]'] + g * self.neg_share()

    def neg_share(self):
        """E[z^2 1(z < 0)] for the innovation distribution (0.5 if symmetric); enters the GJR persistence."""
        d = self.res.model.distribution
        if d.name.startswith('Standardized Skew'):
            from scipy import integrate
            par = self.res.params[d.parameter_names()].values
            pdf = lambda x: float(np.exp(d.loglikelihood(par, np.array([x]), np.array([1.0]), individual=True))[0])
            return float(integrate.quad(lambda x: x ** 2 * pdf(x), -np.inf, 0, limit=200)[0])
        return 0.5

    def uncond_var(self):
        p = self.params
        if self.vol == 'EGARCH':
            return np.nan
        return p['omega'] / (1 - self.persistence)

    def forecast_var(self, horizon):
        """Daily variance forecasts for t+1, ..., t+h, in %^2."""
        f = self.res.forecast(horizon=horizon, reindex=False, method='analytic' if self.vol != 'EGARCH' else 'simulation',
                              simulations=2000)
        return f.variance.iloc[-1].values / self.c ** 2


def scale_for(r):
    """Rescaling factor (a power of 10) that makes the standard deviation of order 1."""
    s = r.std()
    return 10.0 if s < 0.5 else 1.0


def fit_garch(r, vol='GARCH', dist='t', mean='Constant', last_obs=None, c=None):
    c = scale_for(r) if c is None else c
    am = arch_model(c * r, mean=mean, dist=dist, **VOL_SPEC[vol])
    res = am.fit(disp='off', last_obs=last_obs, options={'maxiter': 1000})
    return Fit(res, c, r, vol)


def half_life(persistence):
    """Number of days after which half of a variance shock has gone."""
    return np.log(0.5) / np.log(persistence) if 0 < persistence < 1 else np.inf


def news_impact(fit, eps, sigma2_bar=None):
    """sigma^2_t as a function of the shock eps_{t-1}, with sigma^2_{t-1} fixed at the sample variance."""
    p = fit.params
    s2 = fit.r.var() if sigma2_bar is None else sigma2_bar
    if fit.vol == 'EGARCH':
        z = eps / np.sqrt(s2)
        return np.exp(p['omega'] + p['alpha[1]'] * (np.abs(z) - np.sqrt(2 / np.pi)) + p['gamma[1]'] * z
                      + p['beta[1]'] * np.log(s2))
    g = p.get('gamma[1]', 0.0)
    return p['omega'] + (p['alpha[1]'] + g * (eps < 0)) * eps ** 2 + p['beta[1]'] * s2


def sign_bias_test(z, eps):
    """Engle-Ng (1993, eq. 18): z_t^2 on S-_{t-1}, S-_{t-1} eps_{t-1}, S+_{t-1} eps_{t-1}, with eps_{t-1} the
    UNSTANDARDISED residual and z_t the standardised residual; t statistics and the joint test T R^2 ~ chi2(3)."""
    d = pd.concat([pd.Series(z), pd.Series(eps)], axis=1, keys=['z', 'e']).dropna()
    z, e = d['z'].values, d['e'].values
    y = z[1:] ** 2
    el = e[:-1]
    sneg = (el < 0).astype(float)
    X = np.column_stack([np.ones_like(el), sneg, sneg * el, (1 - sneg) * el])
    ols = sm.OLS(y, X).fit()
    joint = len(y) * ols.rsquared
    return {'sign_t': ols.tvalues[1], 'neg_size_t': ols.tvalues[2], 'pos_size_t': ols.tvalues[3],
            'joint': joint, 'joint_p': 1 - stats.chi2.cdf(joint, 3)}


def ljung_box(x, lags=10):
    from statsmodels.stats.diagnostic import acorr_ljungbox
    lb = acorr_ljungbox(pd.Series(np.asarray(x)).dropna(), lags=[lags])
    return float(lb['lb_stat'].iloc[0]), float(lb['lb_pvalue'].iloc[0])


def arch_lm(x, lags=5):
    from statsmodels.stats.diagnostic import het_arch
    lm, lmp, _, _ = het_arch(pd.Series(np.asarray(x)).dropna(), nlags=lags)
    return float(lm), float(lmp)


def ewma_variance(r, lam=0.94, init=250):
    """EWMA forecast for day t+1 (aligned at t+1): s2_{t+1} = lam s2_t + (1 - lam) r_t^2."""
    x = r.values
    s2 = np.empty(len(x) + 1)
    s2[0] = np.mean(x[:init] ** 2)
    for t in range(len(x)):
        s2[t + 1] = lam * s2[t] + (1 - lam) * x[t] ** 2
    return pd.Series(s2[1:-1], index=r.index[1:])


def qlike(proxy, h):
    """QLIKE loss (Patton, 2011), the form used for comparisons: proxy/h + ln h."""
    return proxy / h + np.log(h)


def mz_regression(proxy, h, lags=5):
    """Mincer-Zarnowitz: proxy = a + b h + u, HAC errors; Wald test of a = 0, b = 1."""
    X = sm.add_constant(np.asarray(h))
    ols = sm.OLS(np.asarray(proxy), X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    w = ols.wald_test((np.eye(2), np.array([0.0, 1.0])), scalar=True)
    return {'a': ols.params[0], 'b': ols.params[1], 'a_se': ols.bse[0], 'b_se': ols.bse[1],
            'r2': ols.rsquared, 'wald': float(w.statistic), 'wald_p': float(w.pvalue)}


def dm_test(loss_a, loss_b, lags=5):
    """Diebold-Mariano: d = L_a - L_b; HAC t statistic for the mean of d (negative: A better)."""
    d = np.asarray(loss_a) - np.asarray(loss_b)
    ols = sm.OLS(d, np.ones_like(d)).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    t = float(ols.tvalues[0])
    return {'mean_diff': float(d.mean()), 't': t, 'p': 2 * (1 - stats.norm.cdf(abs(t)))}


LBL = {'T': 'Forecast origin', 'rT': 'Return on the last day r_T (%)', 'eps': 'Shock eps_T = r_T - mu', 's2T': 'Variance sigma_T^2', 's2n': "Tomorrow's variance sigma_{T+1}^2", 'uv': 'Long-run variance', 'h10': 'Variance forecast for day T+10', 'sum10': '10-day variance (sum of the forecasts)', 'vol10_pct': '10-day volatility (%)', 'cond': 'Fourth-moment condition (alpha+beta)^2 + 2 alpha^2', 'K': 'Kurtosis K', 'corr': 'Correlation of GARCH-t volatility and the VIX', 'mean_vix': 'Mean VIX (%)', 'mean_garch': 'Mean GARCH-t volatility (%)', 'max_garch': 'Highest GARCH-t volatility (%)', 'max_garch_date': 'Date of the highest GARCH-t volatility', 'min_garch': 'Lowest GARCH-t volatility (%)', 'min_garch_date': 'Date of the lowest GARCH-t volatility', 'share_vix_above': 'Share of days with the VIX above GARCH-t', 'garch_max': 'Peak GARCH-t volatility, 2020 (%)', 'gjr_max': 'Peak GJR-t volatility, 2020 (%)', 'vix_max': 'Peak VIX, 2020', 'vix_max_date': 'Date of the VIX peak', 'garch_max_date': 'Date of the GARCH-t peak', 'rmin': 'Lowest return, 2020 (%)', 'rmin_date': 'Date of the lowest return', 'vix_0221': 'VIX on 21 Feb 2020', 'vix_0224': 'VIX on 24 Feb 2020', 'garch_0224': 'GARCH-t volatility on 24 Feb 2020 (%)', 'garch_0225': 'GARCH-t volatility on 25 Feb 2020 (%)', 'r_0224': 'Return on 24 Feb 2020 (%)', 'z_kurt': 'Excess kurtosis of z', 'z_skew': 'Skewness of z', 'jb': 'Jarque-Bera statistic', 'z_min': 'Smallest z', 'z_min_date': 'Date of the smallest z', 'nu_t': 'Student-t degrees of freedom nu (GARCH-t)', 'r_kurt': 'Excess kurtosis of the returns', 'n_below4': 'Residuals below -4', 'exp_below4_normal': 'Expected below -4, Normal', 'exp_below4_t': 'Expected below -4, Student-t', 'skewt_eta': 'Skewed-t degrees of freedom nu (GJR)', 'skewt_lambda': 'Skewed-t asymmetry lambda (GJR)', 'r_1': 'ACF of r_t, lag 1', 'r2_1': 'ACF of r_t^2, lag 1', 'r2_10': 'ACF of r_t^2, lag 10', 'r2_50': 'ACF of r_t^2, lag 50', 'band': '95% band', 'n_r2_sig': 'Significant lags of r_t^2', 'n_r_sig': 'Significant lags of r_t', 'kurt_garch': 'Excess kurtosis, simulated GARCH', 'acf1_x2': 'ACF of squared returns, lag 1', 'kurt_theory': 'Theoretical excess kurtosis', 'uvar': 'Unconditional variance', 's2bar': 'Long-run variance', 'gjr_m3': 'GJR variance after a -3% shock', 'gjr_p3': 'GJR variance after a +3% shock', 'eg_m3': 'EGARCH variance after a -3% shock', 'eg_p3': 'EGARCH variance after a +3% shock', 'g_pm3': 'GARCH variance after a shock of +/-3%', 'emp_1': 'ACF of |r_t|, lag 1, data', 'emp_100': 'ACF of |r_t|, lag 100, data', 'emp_250': 'ACF of |r_t|, lag 250, data', 'garch_1': 'ACF of |r_t|, lag 1, GARCH', 'garch_100': 'ACF of |r_t|, lag 100, GARCH', 'garch_250': 'ACF of |r_t|, lag 250, GARCH', 'fig_1': 'ACF of |r_t|, lag 1, FIGARCH', 'fig_100': 'ACF of |r_t|, lag 100, FIGARCH', 'fig_250': 'ACF of |r_t|, lag 250, FIGARCH', 'd': 'FIGARCH memory parameter d', 'd_se': 's.e. of d', 'fig_bic': 'BIC, FIGARCH', 'garch_bic': 'BIC, GARCH', 'kappa_z': 'Kurtosis of z', 'theory_ratio': 'Theoretical s.e. ratio sqrt((kappa_z - 1)/2)', 'ratio_alpha': 'Observed robust/classical ratio, alpha', 'ratio_beta': 'Observed robust/classical ratio, beta', 'kappa_half': 'Kurtosis of z, first half of the sample', 'alpha': 'alpha', 'beta': 'beta', 'gamma': 'gamma', 'lr': 'LR statistic', 'p_mix': 'p-value, mixture 1/2 chi2(0) + 1/2 chi2(1)', 'nu': 'nu', 'crit_mix': '5% critical value, mixture', 'crit_chi2': '5% critical value, chi2(1)', 'gjr_alpha': 'alpha of GJR-t, S&P 500', 'pers': 'Persistence p = alpha + beta', 'se_p': 's.e. of p', 's2': 'Long-run variance', 'se_s2': 's.e. of the long-run variance', 'vol': 'Long-run volatility (% p.a.)', 'se_vol': 's.e. of the long-run volatility', 'hl': 'Half-life (days)', 'dh': 'dh/dp', 'se_hl': 's.e. of the half-life (delta method)', 'p_lo': '95% interval for p, lower end', 'p_hi': '95% interval for p, upper end', 'hl_lo': 'Half-life at the lower end of p', 'rho1': 'rho_1(eps^2), model', 'rho10': 'rho_10(eps^2), model', 'rho1_sample': 'rho_1(eps^2), sample', 'rho10_sample': 'rho_10(eps^2), sample', 'cond3': 'Condition, estimates rounded to 3 decimals', 'K3': 'Kurtosis K, estimates rounded to 3 decimals', 'rho13': 'rho_1, rounded estimates', 'M': 'Number of lags M', 'q_lb': 'Ljung-Box Q on z^2 (naive)', 'p_lb': 'p-value, naive', 'q_lm': 'Li-Mak corrected statistic', 'p_lm': 'p-value, Li-Mak', 'r1': 'First autocorrelation of z^2', 'c11': 'C[1,1]', 'c55': 'C[5,5]', 'cMM': 'C[M,M]', 'trace_loss': 'M - trace(C)', 'eta': 'Skewed-t degrees of freedom nu', 'lam': 'Skewed-t asymmetry lambda', 'm_neg': 'E[z^2 1(z<0)]', 'p_neg': 'P(z<0)', 'pers_sym': 'Persistence with 1/2', 'pers_skew': 'Persistence with E[z^2 1(z<0)]', 'll_garch': 'Log-likelihood, GARCH', 'll_gjr': 'Log-likelihood, GJR', 'aic_garch': 'AIC, GARCH', 'aic_gjr': 'AIC, GJR', 'bic_garch': 'BIC, GARCH', 'bic_gjr': 'BIC, GJR', 'lr_p': 'LR p-value, chi2(1)', 'gamma_t': 'Robust t of gamma', 'B': 'Bootstrap paths B', 'est': 'Estimated persistence p', 'pers_lo': '2.5% percentile of p*', 'pers_hi': '97.5% percentile of p*', 'pers_sd': 'Standard deviation of p*', 'pers_mean': 'Mean of p*', 'hl_med': 'Median half-life (days)', 'hl_hi': '97.5% percentile of the half-life', 'share_inf': 'Share of draws with an infinite half-life', 'share_ge_0999': 'Share of draws with p* >= 0.999', 'se_robust_sum': 'Robust s.e. of alpha + beta', 'alpha0': 'alpha under H0 (IGARCH-t)', 'omega0': 'omega under H0', 'nu0': 'nu under H0', 'p_chi2': 'p-value, naive chi2(1)', 'p_boot_pers': 'Bootstrap p-value of alpha + beta', 'p_boot_lr': 'Bootstrap p-value of LR', 'share_bound': 'Share of re-estimates with alpha + beta >= 0.999', 'lr_q95': '95% quantile of LR*', 'pers_q05': '5% quantile of p* under H0'}


def show(title, d, skip=()):
    """Print a dictionary of results as labelled lines."""
    print(title)
    for k, v in d.items():
        if k in skip or isinstance(v, (dict, list)):
            continue
        s = f'{v:.4f}' if isinstance(v, float) else str(v)
        print(f'  {LBL.get(k, k)}: {s}')


## Seminar functions

- Model grid, asymmetry table and out-of-sample forecasts (from the lecture), and the functions used in Parts A–C.

In [ ]:
MODELS_OOS = [('GARCH', 'normal', 'GARCH-N'), ('GARCH', 't', 'GARCH-t'), ('GJR', 't', 'GJR-t'), ('EGARCH', 't', 'EGARCH-t')]
OUT = {}
DRAWS = {}   # bootstrap draws (B7, B8)


def model_grid():
    rows, fits = [], {}
    for k in ORDER:
        r = rets[k]
        for vol in VOL_SPEC:
            for dist in DISTS:
                f = fit_garch(r, vol, dist)
                fits[(k, vol, dist)] = f
                p = f.params
                rows.append(dict(market=k, vol=vol, dist=dist, loglik=f.loglik, aic=f.aic, bic=f.bic,
                                 k=len(p), converged=f.converged, persistence=f.persistence,
                                 **{f'p_{n}': v for n, v in p.items()}))
    g = pd.DataFrame(rows)
    g['dbic'] = g['bic'] - g.groupby('market')['bic'].transform('min')
    g.to_csv(os.path.join(TABLE_DIR, 'ch5_model_grid.csv'), index=False)
    return g, fits


def garch_table(fits):
    """GARCH(1,1)-t on each series: parameters, robust and classical errors, persistence, half-life."""
    rows = []
    for k in ORDER:
        f = fits[(k, 'GARCH', 't')]
        p, se = f.params, f.se
        classic = f.res.model.fit(disp='off', cov_type='classic', starting_values=f.res.params.values).std_err
        pers = p['alpha[1]'] + p['beta[1]']
        uv = f.uncond_var()
        rows.append(dict(market=k, mu=p['mu'], omega=p['omega'], alpha=p['alpha[1]'], beta=p['beta[1]'], nu=p['nu'],
                         se_alpha=se['alpha[1]'], se_beta=se['beta[1]'], se_nu=se['nu'],
                         se_alpha_classic=classic['alpha[1]'], se_beta_classic=classic['beta[1]'],
                         persistence=pers, half_life=half_life(pers),
                         uncond_ann_vol=np.sqrt(uv * PPY[k]) if pers < 0.999 else np.nan,
                         last_ann_vol=f.sigma.iloc[-1] * np.sqrt(PPY[k]), loglik=f.loglik, N=f.nobs, scale=f.c))
    t = pd.DataFrame(rows).set_index('market')
    t.to_csv(os.path.join(TABLE_DIR, 'ch5_garch_table.csv'))
    return t


def asym_table(fits):
    rows = []
    for k in ORDER:
        g, j, e = fits[(k, 'GARCH', 't')], fits[(k, 'GJR', 't')], fits[(k, 'EGARCH', 't')]
        lr = 2 * (j.loglik - g.loglik)
        rows.append(dict(market=k, gjr_alpha=j.params['alpha[1]'], gjr_gamma=j.params['gamma[1]'],
                         gjr_gamma_t=j.tvalues['gamma[1]'], gjr_beta=j.params['beta[1]'],
                         gjr_persistence=j.persistence, lr=lr, lr_p=1 - stats.chi2.cdf(lr, 1),
                         eg_alpha=e.params['alpha[1]'], eg_gamma=e.params['gamma[1]'], eg_gamma_t=e.tvalues['gamma[1]'],
                         eg_beta=e.params['beta[1]'], bic_garch=g.bic, bic_gjr=j.bic, bic_egarch=e.bic))
    t = pd.DataFrame(rows).set_index('market')
    t.to_csv(os.path.join(TABLE_DIR, 'ch5_asym_table.csv'))
    return t


def oos_forecasts(k='sp500', first_year=2016, H=22):
    r = rets[k]
    idx = r.index
    last_year = idx[-1].year
    out1 = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS}
    outH = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS if lab != 'EGARCH-t'}
    for y in range(first_year, last_year + 1):
        start = idx[idx < f'{y}-01-01'][-1]
        end = idx[idx <= f'{y}-12-31'][-1]
        orig = idx[(idx >= start) & (idx < end)]
        for vol, dist, lab in MODELS_OOS:
            f = fit_garch(r, vol, dist, last_obs=idx[idx >= f'{y}-01-01'][0])
            hh = 1 if vol == 'EGARCH' else H
            fc = f.res.forecast(horizon=hh, start=start, reindex=False).variance / f.c ** 2
            fc = fc.loc[orig]
            pos = idx.get_indexer(fc.index)
            out1[lab].iloc[pos + 1] = fc.iloc[:, 0].values
            if hh == H and lab in outH:
                outH[lab].loc[fc.index] = fc.sum(axis=1).values      # cumulative variance over t+1..t+H, at origin t
    ew = ewma_variance(r)
    hist = (r ** 2).rolling(20).mean().shift(1)
    out1['EWMA'] = ew.reindex(idx)
    out1['Hist-20'] = hist
    outH['EWMA'] = (ew.shift(-1) * H).reindex(idx)       # the EWMA forecast is flat across horizons
    f1 = pd.DataFrame(out1).loc[f'{first_year}-01-01':].dropna()
    fH = pd.DataFrame(outH).dropna()
    return f1, fH


def a1_recursion():
    om, al, be = 0.05, 0.10, 0.85
    eps = [-2.0, 0.5, 3.0, -1.0]
    s2 = [om / (1 - al - be)]
    for e in eps:
        s2.append(om + al * e ** 2 + be * s2[-1])
    uv = om / (1 - al - be)
    last = s2[-1]
    fc = [uv + (al + be) ** (h - 1) * (last - uv) for h in range(1, 11)]
    OUT['a1'] = {'omega': om, 'alpha': al, 'beta': be, 'eps': eps, 's2': s2, 'uv': uv, 'fc': fc,
                 'hl': half_life(al + be), 'sum5': float(np.sum(fc[:5])), 'sum10': float(np.sum(fc))}


def lr_tests():
    at = pd.read_csv(os.path.join(TABLE_DIR, 'ch5_asym_table.csv'), index_col=0)
    grid = pd.read_csv(os.path.join(TABLE_DIR, 'ch5_model_grid.csv'))
    res = {}
    for k in ['sp500', 'bet']:
        sub = grid[(grid.market == k) & (grid.dist == 't')].set_index('vol')
        res[k] = {'ll_garch': sub.loc['GARCH', 'loglik'], 'll_gjr': sub.loc['GJR', 'loglik'],
                  'aic_garch': sub.loc['GARCH', 'aic'], 'aic_gjr': sub.loc['GJR', 'aic'],
                  'bic_garch': sub.loc['GARCH', 'bic'], 'bic_gjr': sub.loc['GJR', 'bic'],
                  'lr': at.loc[k, 'lr'], 'lr_p': at.loc[k, 'lr_p'], 'gamma': at.loc[k, 'gjr_gamma'],
                  'gamma_t': at.loc[k, 'gjr_gamma_t'], 'alpha': at.loc[k, 'gjr_alpha'], 'beta': at.loc[k, 'gjr_beta']}
    OUT['lr'] = res


def btc_innovations():
    r = rets['btc']
    out = {}
    for d in ['normal', 't', 'skewt', 'ged']:
        f = fit_garch(r, 'GARCH', d)
        out[d] = {'loglik': f.loglik, 'aic': f.aic, 'bic': f.bic,
                  **{k: float(v) for k, v in f.params.items()}, 'persistence': float(f.persistence)}
        if d == 'skewt':
            out[d]['lambda_se'] = float(f.se['lambda'])
            out[d]['lambda_t'] = float(f.tvalues['lambda'])
        if d == 't':
            out[d]['nu_se'] = float(f.se['nu'])
    out['lr_skew'] = 2 * (out['skewt']['loglik'] - out['t']['loglik'])
    out['lr_skew_p'] = float(1 - stats.chi2.cdf(max(out['lr_skew'], 0), 1))
    z = fit_garch(r, 'GARCH', 't').z.dropna()
    out['z_skew'] = float(stats.skew(z))
    out['z_kurt'] = float(stats.kurtosis(z))
    OUT['btc'] = out


def forecast_comparison(k):
    r = rets[k]
    f1, _ = oos_forecasts(k, 2016, 1)
    proxy = (r ** 2).reindex(f1.index)
    base = qlike(proxy, f1['GARCH-t'])
    rows = {}
    for m in f1.columns:
        L = qlike(proxy, f1[m])
        dm = dm_test(L, base) if m != 'GARCH-t' else {'mean_diff': 0.0, 't': float('nan'), 'p': float('nan')}
        mz = mz_regression(proxy, f1[m])
        rows[m] = {'qlike': float(L.mean()), 'dm_diff': dm['mean_diff'], 'dm_t': dm['t'], 'dm_p': dm['p'],
                   'mz_a': mz['a'], 'mz_b': mz['b'], 'mz_b_se': mz['b_se'], 'mz_r2': mz['r2'], 'mz_wald_p': mz['wald_p']}
    OUT[f'fc_{k}'] = {'n': len(f1), 'start': str(f1.index[0].date()), 'models': rows}


def tgarch_negll(theta, r, igarch=False):
    """Minus the log-likelihood of GARCH(1,1)-t with constant mean mu.
    Parametrisation: p = alpha + beta in [0, 1], s = alpha / p in [0, 1]; IGARCH: p = 1.
    Start as in arch: sigma^2_0 = eps^2_0 = EWMA-weighted mean (0.94) of the first 75 squares."""
    if igarch:
        mu, om, s, nu = theta
        p = 1.0
    else:
        mu, om, p, s, nu = theta
    e = r - mu
    al, be = p * s, p * (1 - s)
    w = 0.94 ** np.arange(75)
    bc = np.sum(e[:75] ** 2 * w) / w.sum()
    drive = om + al * np.concatenate(([bc], e[:-1] ** 2))
    s2 = lfilter([1.0], [1.0, -be], drive, zi=[be * bc])[0]
    z2 = e ** 2 / s2
    ll = (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2)) - 0.5 * np.log(s2)
          - (nu + 1) / 2 * np.log1p(z2 / (nu - 2)))
    return -ll.sum()


def fit_tgarch(r, igarch=False, start=None):
    """GARCH(1,1)-t (or IGARCH(1,1)-t, alpha + beta = 1) by our own likelihood, with the restriction alpha + beta <= 1.
    The unrestricted model is also started from the IGARCH solution: l_GARCH >= l_IGARCH, so LR >= 0."""
    r = np.asarray(r, dtype=float)
    m, v = np.mean(r), np.var(r)
    opts = {'ftol': 1e-12, 'gtol': 1e-7, 'maxiter': 3000}
    if igarch:
        x0 = [m, 0.01 * v, 0.1, 6.0] if start is None else start
        bnd = [(None, None), (1e-8 * v, 10 * v), (1e-4, 1.0), (2.05, 200.0)]
        o = optimize.minimize(tgarch_negll, x0, args=(r, True), method='L-BFGS-B', bounds=bnd, options=opts)
        mu, om, s, nu = o.x
        return {'mu': mu, 'omega': om, 'alpha': s, 'beta': 1 - s, 'nu': nu, 'pers': 1.0, 'loglik': -o.fun}
    g = fit_tgarch(r, igarch=True)
    bnd = [(None, None), (1e-8 * v, 10 * v), (0.0, 1.0), (1e-4, 1.0), (2.05, 200.0)]
    starts = [[m, 0.01 * v, 0.98, 0.1, 6.0], [g['mu'], g['omega'], 0.999, g['alpha'], g['nu']]]
    if start is not None:
        starts.append(start)
    best = min((optimize.minimize(tgarch_negll, x0, args=(r, False), method='L-BFGS-B', bounds=bnd, options=opts)
                for x0 in starts), key=lambda o: o.fun)
    mu, om, p, s, nu = best.x
    return {'mu': mu, 'omega': om, 'alpha': p * s, 'beta': p * (1 - s), 'nu': nu, 'pers': p,
            'loglik': max(-best.fun, g['loglik']), 'loglik_igarch': g['loglik']}


def sim_tgarch(par, n, B, rng, s2_start, burn=500):
    """B GARCH(1,1)-t paths at once (vectorised over columns); also works for alpha + beta = 1."""
    nu = par['nu']
    z = rng.standard_t(nu, size=(n + burn, B)) * np.sqrt((nu - 2) / nu)
    s2 = np.full(B, s2_start)
    out = np.empty((n + burn, B))
    for t in range(n + burn):
        e = np.sqrt(s2) * z[t]
        out[t] = e
        s2 = par['omega'] + par['alpha'] * e ** 2 + par['beta'] * s2
    return par['mu'] + out[burn:]


def param_bootstrap(k, B=999):
    """Parametric bootstrap: B paths from the fitted GARCH(1,1)-t, re-estimated; percentile intervals."""
    from joblib import Parallel, delayed
    r = rets[k]
    f = fit_garch(r, 'GARCH', 't')
    p = f.params
    par = {'mu': p['mu'], 'omega': p['omega'], 'alpha': p['alpha[1]'], 'beta': p['beta[1]'], 'nu': p['nu']}
    rng = np.random.default_rng(SEED)
    sims = sim_tgarch(par, len(r), B, rng, r.var())
    st = [par['mu'], par['omega'], par['alpha'] + par['beta'], par['alpha'] / (par['alpha'] + par['beta']), par['nu']]
    fits = Parallel(n_jobs=-1)(delayed(fit_tgarch)(sims[:, b], False, st) for b in range(B))
    pers = np.array([x['pers'] for x in fits])
    hls = np.array([half_life(x) for x in pers])
    hq = np.sort(hls)                                          # np.inf kept: persistence at the bound 1
    DRAWS[f'boot_pers_{k}'] = pers
    est = p['alpha[1]'] + p['beta[1]']
    OUT[f'boot_{k}'] = {'B': B, 'est': float(est), 'hl': float(half_life(est)),
                        'pers_lo': float(np.quantile(pers, 0.025)), 'pers_hi': float(np.quantile(pers, 0.975)),
                        'pers_sd': float(pers.std()), 'pers_mean': float(pers.mean()),
                        'hl_lo': float(np.quantile(hq, 0.025)), 'hl_med': float(np.median(hq)),
                        # h(p) is increasing: the upper end is h(97.5% quantile of p), infinite if p = 1
                        'hl_hi': float(half_life(np.quantile(pers, 0.975))), 'share_inf': float((pers >= 1 - 1e-6).mean()),
                        'share_ge_0999': float((pers >= 0.999).mean()),
                        'se_robust_sum': float(np.sqrt(f.res.param_cov.loc[['alpha[1]', 'beta[1]'], ['alpha[1]', 'beta[1]']].values.sum()))}
    return pers, hls


def _null_rep(y):
    ub = fit_tgarch(y)
    return ub['pers'], 2 * (ub['loglik'] - ub['loglik_igarch'])


def igarch_test(k, B=999):
    """Test of H0: alpha + beta = 1 (IGARCH) by parametric bootstrap UNDER H0 (Andrews, 2000: a percentile interval
    truncated at 1 is not a valid test). Statistics: the estimated persistence and LR = 2(l_GARCH - l_IGARCH)."""
    from joblib import Parallel, delayed
    r = rets[k]
    u = fit_tgarch(r)
    g0 = fit_tgarch(r, igarch=True)
    lr = 2 * (u['loglik'] - g0['loglik'])
    rng = np.random.default_rng(SEED + 1)
    sims = sim_tgarch(g0, len(r), B, rng, r.var())
    res = Parallel(n_jobs=-1)(delayed(_null_rep)(sims[:, b]) for b in range(B))
    pb, lrb = np.array([x[0] for x in res]), np.array([x[1] for x in res])
    DRAWS[f'null_pers_{k}'], DRAWS[f'null_lr_{k}'] = pb, lrb
    OUT[f'igarch_{k}'] = {'B': B, 'pers': u['pers'], 'alpha0': g0['alpha'], 'omega0': g0['omega'], 'nu0': g0['nu'],
                          'lr': lr, 'p_chi2': float(stats.chi2.sf(lr, 1)), 'p_mix': float(0.5 * stats.chi2.sf(lr, 1)),
                          'p_boot_pers': float((1 + np.sum(pb <= u['pers'])) / (B + 1)),
                          'p_boot_lr': float((1 + np.sum(lrb >= lr)) / (B + 1)),
                          'share_bound': float(np.mean(pb >= 0.999)), 'lr_q95': float(np.quantile(lrb, 0.95)),
                          'pers_q05': float(np.quantile(pb, 0.05))}
    return pb


def fig_bootstrap(boots, nulls):
    """Bootstrap distribution of the persistence: from the fitted model and under H0 (IGARCH)."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax, (k, lab, col) in zip(axes, [('sp500', 'S&P 500', MainBlue), ('bet', 'BET', IDAred)]):
        bins = np.linspace(min(boots[k].min(), nulls[k].min()), 1.0, 40)
        ax.hist(boots[k], bins=bins, color=col, alpha=0.75, label=f'{lab}: fitted GARCH-t (B = {len(boots[k])})')
        ax.hist(nulls[k], bins=bins, color=Amber, alpha=0.6, label=f'{lab}: IGARCH-t under H0 (B = {len(nulls[k])})')
        ax.axvline(OUT[f'boot_{k}']['est'], color='black', lw=1, label='Estimate on the data')
        ax.set_xlabel(r'$\hat\alpha+\hat\beta$')
        legend_outside_bottom(ax, ncol=1, y=-0.2)
    fig.tight_layout()
    save_fig('ch5_sem_bootstrap')


def join_bet_sp():
    """Align PRICES on common days, then take returns; the US shock relevant for the BET on day t
    is the S&P 500 return of the previous common day (the US market closes after the BVB)."""
    px = pd.concat([load_close('bet'), load_close('sp500')], axis=1, join='inner').dropna()
    ret = 100 * np.log(px).diff().dropna()
    df = pd.DataFrame({'bet': ret['bet'], 'us_lag': ret['sp500'].shift(1)}).dropna()
    return df


def spill_negll(theta, y, x, delta_free=True):
    mu, phi, lw, la, lb, ld, lnu = theta
    om, al, be = np.exp(lw), np.exp(la), np.exp(lb)
    de = np.exp(ld) if delta_free else 0.0
    nu = 2.05 + np.exp(lnu)
    if al + be >= 0.9999:
        return 1e10
    e = y - mu - phi * x
    n = len(y)
    # s2_t = beta s2_{t-1} + (omega + alpha e_{t-1}^2 + delta u_{t-1}^2): a linear recursive filter;
    # x_t = u_{t-1} (the S&P 500 return of the previous common day), so the variance regressor is x_t^2
    drive = np.empty(n)
    drive[0] = np.var(y) * (1 - be)
    drive[1:] = om + al * e[:-1] ** 2 + de * x[1:] ** 2
    s2 = lfilter([1.0], [1.0, -be], drive)
    z2 = e ** 2 / s2
    ll = (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2)) - 0.5 * np.log(s2)
          - (nu + 1) / 2 * np.log1p(z2 / (nu - 2)))
    return -ll.sum()


def fit_spill(df, delta_free=True):
    y, x = df['bet'].values, df['us_lag'].values
    th0 = np.array([0.05, 0.1, np.log(0.03), np.log(0.15), np.log(0.8), np.log(0.02), np.log(3.0)])
    best = None
    for start_ld in [np.log(0.02), np.log(0.1)]:
        th0[5] = start_ld
        o = optimize.minimize(spill_negll, th0, args=(y, x, delta_free), method='Nelder-Mead',
                              options={'maxiter': 20000, 'maxfev': 20000, 'xatol': 1e-7, 'fatol': 1e-7})
        o = optimize.minimize(spill_negll, o.x, args=(y, x, delta_free), method='L-BFGS-B')
        if best is None or o.fun < best.fun:
            best = o
    th = best.x
    par = {'mu': th[0], 'phi': th[1], 'omega': np.exp(th[2]), 'alpha': np.exp(th[3]), 'beta': np.exp(th[4]),
           'delta': np.exp(th[5]) if delta_free else 0.0, 'nu': 2.05 + np.exp(th[6]), 'loglik': -best.fun}
    return par


def spill_path(df, par):
    """Conditional variance of the BET and the part due to the previous-day US shock."""
    y, x = df['bet'].values, df['us_lag'].values
    e = y - par['mu'] - par['phi'] * x
    n = len(y)
    drive = np.empty(n)
    drive[0] = np.var(y) * (1 - par['beta'])
    drive[1:] = par['omega'] + par['alpha'] * e[:-1] ** 2 + par['delta'] * x[1:] ** 2
    s2 = lfilter([1.0], [1.0, -par['beta']], drive)
    us = np.zeros(n)
    us[1:] = par['delta'] * x[1:] ** 2
    # cumulative contribution of the US term through the beta recursion: sum_j beta^j delta u_{t-1-j}^2
    us_total = lfilter([1.0], [1.0, -par['beta']], us)
    return pd.DataFrame({'s2': s2, 'us': us_total}, index=df.index)


def part_c():
    df = join_bet_sp()
    res = {'n': len(df), 'start': str(df.index[0].date()), 'end': str(df.index[-1].date()),
           'corr_same': float(np.corrcoef(df['bet'], df['us_lag'])[0, 1])}
    for name, sub in [('full', df), ('pre2013', df.loc[:'2012-12-31']), ('post2013', df.loc['2013-01-01':])]:
        u = fit_spill(sub, True)
        rr = fit_spill(sub, False)
        lr = 2 * (u['loglik'] - rr['loglik'])
        # delta >= 0: under the null the parameter is on the boundary -> 0.5 chi2(0) + 0.5 chi2(1)
        p = 0.5 * (1 - stats.chi2.cdf(max(lr, 0), 1))
        x2 = (sub['us_lag'] ** 2).mean()
        share = u['delta'] * x2 / (u['omega'] + u['delta'] * x2)
        res[name] = {'u': u, 'r': rr, 'lr': lr, 'p': p, 'n': len(sub), 'share_intercept': share}
        if name == 'full':
            path = spill_path(sub, u)
    share = (path['us'] / path['s2']).clip(0, 1)
    res['share_mean'] = float(share.mean())
    m = share.resample('YE').mean()
    res['share_max_year'] = int(m.idxmax().year)
    res['share_max'] = float(m.max())
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(share.index, share.rolling(63).mean(), color=IDAred, lw=0.9,
            label='Model-based share of BET conditional variance generated by the lagged S&P 500 term (3-month average)')
    ax.set_ylabel('Share')
    ax.set_ylim(0, None)
    legend_outside_bottom(ax, ncol=1, y=-0.15)
    save_fig('ch5_sem_spillover')
    OUT['partc'] = res


from scipy import integrate
from arch.bootstrap import MCS
INF = {}


def qmle_efficiency():
    """Gaussian QMLE: avar = (kappa_z - 1) J^{-1}; classical (inverse Hessian) = 2 J^{-1}; s.e. ratio sqrt((kappa_z-1)/2)."""
    out = {}
    for k in ['sp500', 'btc']:
        f = fit_garch(rets[k], 'GARCH', 'normal')
        cl = f.res.model.fit(disp='off', cov_type='classic', starting_values=f.res.params.values)
        z = f.z.dropna()
        kz = float(stats.kurtosis(z, fisher=False))
        out[k] = {'kappa_z': kz, 'theory_ratio': float(np.sqrt((kz - 1) / 2)),
                  'ratio_alpha': float(f.se['alpha[1]'] / cl.std_err['alpha[1]']),
                  'ratio_beta': float(f.se['beta[1]'] / cl.std_err['beta[1]']),
                  'alpha': float(f.params['alpha[1]']), 'beta': float(f.params['beta[1]'])}
        # stability of the sample kurtosis: first half vs full sample
        out[k]['kappa_half'] = float(stats.kurtosis(z.iloc[:len(z) // 2], fisher=False))
    INF['qmle'] = out


def boundary_tests():
    """LR for Normal (1/nu = 0) against Student-t: 1/nu >= 0, so the limit law is 1/2 chi2(0) + 1/2 chi2(1)."""
    out = {'crit_mix': float(stats.chi2.ppf(0.90, 1)), 'crit_chi2': float(stats.chi2.ppf(0.95, 1))}
    for k in ['sp500', 'btc']:
        fn, ft = fit_garch(rets[k], 'GARCH', 'normal'), fit_garch(rets[k], 'GARCH', 't')
        lr = 2 * (ft.loglik - fn.loglik)
        out[k] = {'lr': float(lr), 'p_mix': float(0.5 * stats.chi2.sf(lr, 1)), 'nu': float(ft.params['nu'])}
    g = fit_garch(rets['sp500'], 'GJR', 't')
    out['gjr_alpha'] = float(g.params['alpha[1]'])
    INF['boundary'] = out


def delta_method():
    """Delta method from the robust covariance of (omega, alpha, beta), GARCH(1,1)-t, S&P 500."""
    f = fit_garch(rets['sp500'], 'GARCH', 't')
    p = f.params
    V = f.res.param_cov.loc[['omega', 'alpha[1]', 'beta[1]'], ['omega', 'alpha[1]', 'beta[1]']].values / f.c ** 2
    om, a, b = p['omega'], p['alpha[1]'], p['beta[1]']
    pers = a + b
    se_p = float(np.sqrt(V[1:, 1:].sum()))
    s2 = om / (1 - pers)
    g = np.array([1 / (1 - pers), om / (1 - pers) ** 2, om / (1 - pers) ** 2])
    se_s2 = float(np.sqrt(g @ V @ g))
    vol = np.sqrt(PPY['sp500'] * s2)
    se_vol = PPY['sp500'] * se_s2 / (2 * vol)
    hl = half_life(pers)
    dh = -np.log(0.5) / (pers * np.log(pers) ** 2)
    lo, hi = pers - 1.96 * se_p, pers + 1.96 * se_p
    INF['delta'] = {'pers': float(pers), 'se_p': se_p, 's2': float(s2), 'se_s2': se_s2, 'vol': float(vol),
                    'se_vol': float(se_vol), 'hl': float(hl), 'dh': float(dh), 'se_hl': float(dh * se_p),
                    'p_lo': float(lo), 'p_hi': float(hi), 'hl_lo': float(half_life(lo)),
                    'grad_om': float(g[0]), 'grad_ab': float(g[1])}


def moments_garch_n():
    """GARCH(1,1) with Normal innovations: (alpha+beta)^2 + 2 alpha^2, K and rho_1(eps^2), at full precision and rounded."""
    f = fit_garch(rets['sp500'], 'GARCH', 'normal')
    a, b = f.params['alpha[1]'], f.params['beta[1]']

    def mom(a, b):
        c = (a + b) ** 2 + 2 * a ** 2
        K = 3 * (1 - (a + b) ** 2) / (1 - c)
        rho1 = a * (1 - b ** 2 - a * b) / (1 - b ** 2 - 2 * a * b)
        return float(c), float(K), float(rho1)
    c, K, rho1 = mom(a, b)
    c3, K3, rho13 = mom(round(a, 3), round(b, 3))
    e2 = (rets['sp500'] - rets['sp500'].mean()) ** 2
    INF['mom'] = {'alpha': float(a), 'beta': float(b), 'cond': c, 'K': K, 'rho1': rho1,
                  'cond3': c3, 'K3': K3, 'rho13': rho13, 'rho10': float(rho1 * (a + b) ** 9),
                  'dKda': float((K3 - K)), 'rho1_sample': float(e2.autocorr(1)), 'rho10_sample': float(e2.autocorr(10))}


def gjr_skew_persistence():
    """GJR-GARCH with skewed-t innovations (Hansen, 1994), S&P 500: E[z^2 1(z<0)] instead of 1/2."""
    f = fit_garch(rets['sp500'], 'GJR', 'skewt')
    d = f.res.model.distribution
    par = f.res.params[d.parameter_names()].values
    pdf = lambda x: float(np.exp(d.loglikelihood(par, np.array([x]), np.array([1.0]), individual=True))[0])
    m_neg = integrate.quad(lambda x: x ** 2 * pdf(x), -np.inf, 0, limit=200)[0]
    p_neg = integrate.quad(pdf, -np.inf, 0, limit=200)[0]
    p = f.params
    INF['gjrskew'] = {'alpha': float(p['alpha[1]']), 'gamma': float(p['gamma[1]']), 'beta': float(p['beta[1]']),
                      'eta': float(par[0]), 'lam': float(par[1]), 'm_neg': float(m_neg), 'p_neg': float(p_neg),
                      'pers_sym': float(p['alpha[1]'] + p['gamma[1]'] / 2 + p['beta[1]']),
                      'pers_skew': float(p['alpha[1]'] + p['gamma[1]'] * m_neg + p['beta[1]'])}


def oos_rolling(k='sp500', first_year=2016, W=1000):
    """One-day forecasts with a ROLLING window of W days, refit every January (Giacomini-White setting)."""
    r = rets[k]
    idx = r.index
    out = {lab: pd.Series(np.nan, index=idx) for *_, lab in MODELS_OOS}
    for y in range(first_year, idx[-1].year + 1):
        start = idx[idx < f'{y}-01-01'][-1]
        end = idx[idx <= f'{y}-12-31'][-1]
        orig = idx[(idx >= start) & (idx < end)]
        pos0 = idx.get_loc(start)
        sub = r.iloc[pos0 - W + 1:]
        for vol, dist, lab in MODELS_OOS:
            f = fit_garch(sub, vol, dist, last_obs=idx[pos0 + 1])
            fc = f.res.forecast(horizon=1, start=start, reindex=False).variance / f.c ** 2
            fc = fc.loc[orig]
            pos = idx.get_indexer(fc.index)
            out[lab].iloc[pos + 1] = fc.iloc[:, 0].values
    out['EWMA'] = ewma_variance(r).reindex(idx)
    out['Hist-20'] = (r ** 2).rolling(20).mean().shift(1)
    return pd.DataFrame(out).loc[f'{first_year}-01-01':].dropna()


def gw_test(la, lb):
    """Giacomini-White (2006), conditional test, horizon 1: instruments x_t = (1, d_t); GW = n Zbar' Omega^{-1} Zbar ~ chi2(2)."""
    d = np.asarray(la) - np.asarray(lb)
    Z = np.column_stack([d[1:], d[:-1] * d[1:]])
    zb = Z.mean(axis=0)
    Om = Z.T @ Z / len(Z)
    stat = len(Z) * zb @ np.linalg.solve(Om, zb)
    return float(stat), float(stats.chi2.sf(stat, 2))


def gw_mcs(k='sp500'):
    r = rets[k]
    fr = oos_rolling(k)
    proxy = (r ** 2).reindex(fr.index)
    L = pd.DataFrame({m: qlike(proxy, fr[m]) for m in fr.columns})
    rows = {}
    for m in L.columns:
        if m == 'GARCH-t':
            continue
        dm = dm_test(L[m], L['GARCH-t'])
        gw, gp = gw_test(L[m], L['GARCH-t'])
        rows[m] = {'uncond_t': dm['t'], 'gw': gw, 'gw_p': gp, 'mean_diff': dm['mean_diff']}
    # MCS on the expanding-window forecasts (the table of the lecture and seminar)
    f1, _ = oos_forecasts(k, 2016, 1)
    px = (r ** 2).reindex(f1.index)
    Le = pd.DataFrame({m: qlike(px, f1[m]) for m in f1.columns})
    mcs = MCS(Le, size=0.10, reps=10000, block_size=10, method='R', bootstrap='stationary', seed=SEED)
    mcs.compute()
    pv = mcs.pvalues['Pvalue']
    INF[f'gw_{k}'] = {'n': len(fr), 'W': 1000, 'qlike_roll': L.mean().to_dict(), 'tests': rows,
                      'mcs_p': pv.to_dict(), 'mcs_in': list(mcs.included), 'n_mcs': len(Le)}


grid, FITS = model_grid()
gt = garch_table(FITS)
at = asym_table(FITS)

## Part A — derivations

### A1. Multi-step variance forecast: derivation and numbers [Solved]

- Derive $\mathbb{E}_t\sigma_{t+h}^2 = \bar\sigma^2 + p^{h-1}(\sigma_{t+1}^2 - \bar\sigma^2)$ and $V_H = H\bar\sigma^2 + (\sigma_{t+1}^2 - \bar\sigma^2)(1-p^H)/(1-p)$, $p = \alpha+\beta$.
- $\omega = 0.05$, $\alpha = 0.10$, $\beta = 0.85$; shocks $-2$, $0.5$, $3$, $-1$: the path, the forecasts, $V_5$ from the sum and from the closed form, $5\sigma_5^2$ and the half-life.

In [7]:
# Solution
a1_recursion()
a1 = OUT['a1']
p = a1['alpha'] + a1['beta']
print('sigma^2 path:', np.round(a1['s2'], 4))
print('forecasts h = 1..5:', np.round(a1['fc'][:5], 4))
vh = 5 * a1['uv'] + (a1['s2'][-1] - a1['uv']) * (1 - p ** 5) / (1 - p)
print('V_5: sum', round(a1['sum5'], 3), ' closed form', round(vh, 3), ' 5 sigma_5^2', round(5 * a1['s2'][-1], 3), ' half-life', round(a1['hl'], 1))

sigma^2 path: [1.    1.3   1.18  1.953 1.81 ]
forecasts h = 1..5: [1.81   1.7695 1.7311 1.6945 1.6598]
V_5: sum 8.665  closed form 8.665  5 sigma_5^2 9.05  half-life 13.5


### A2. GJR persistence with skewed innovations [Proposed]

- Persistence $\alpha + \gamma\,\mathbb{E}[z^2 1(z<0)] + \beta$; S&P 500 GJR with skewed-t innovations: compute $\mathbb{E}[z^2 1(z<0)]$ numerically. Model: A1.

In [ ]:
# Your code here


### A3. Long-run volatility, half-life and delta-method standard errors [Solved]

- S&P 500 GARCH(1,1)-t: $\sigma^2 = \omega/(1-\alpha-\beta)$, daily and annual volatility, half-life; then delta-method standard errors from the robust covariance of $(\hat\omega, \hat\alpha, \hat\beta)$.

In [9]:
# Solution
p = gt.loc['sp500']
uv = p['omega'] / (1 - p['persistence'])
print('alpha+beta', round(p['persistence'], 4), ' long-run variance', round(uv, 3), ' daily vol', round(np.sqrt(uv), 3))
print('annual vol', round(np.sqrt(uv * PPY['sp500']), 1), ' sample', round(rets['sp500'].std() * np.sqrt(PPY['sp500']), 1))
print('half-life', round(half_life(p['persistence']), 1), 'days')
delta_method()
show('Delta method', INF['delta'], skip=('grad_om', 'grad_ab'))

alpha+beta 0.9947  long-run variance 2.972  daily vol 1.724
annual vol 27.3  sample 19.2
half-life 130.7 days


Delta method
  Persistence p = alpha + beta: 0.9947
  s.e. of p: 0.0047
  Long-run variance: 2.9717
  s.e. of the long-run variance: 2.3499
  Long-run volatility (% p.a.): 27.3336
  s.e. of the long-run volatility: 10.8073
  Half-life (days): 130.7335
  dh/dp: 24788.5443
  s.e. of the half-life (delta method): 115.9302
  95% interval for p, lower end: 0.9855
  95% interval for p, upper end: 1.0039
  Half-life at the lower end of p: 47.6066


### A4. Is the fourth moment finite? [Proposed]

- GARCH-N for the S&P 500: check $(\alpha+\beta)^2 + 2\alpha^2 < 1$ and compute the kurtosis, also with estimates rounded to 3 decimals; then the Student-t condition $(\alpha+\beta)^2 + (\kappa_z - 1)\alpha^2 < 1$ with $\kappa_z = 3(\nu-2)/(\nu-4)$. Model: A1, A3.

In [ ]:
# Your code here


### A5. Robust versus classical variance of the Gaussian QMLE, ARCH(1) [Proposed]

- Derive the score and Hessian of the Gaussian quasi-likelihood; show avar $= \frac{\kappa_z - 1}{2}\times$ the classical one; compare $\sqrt{(\hat\kappa_z - 1)/2}$ with the observed robust / classical ratio (S&P 500, GARCH-N). Model: A3.

In [ ]:
# Your code here


### A6. Autocorrelation of squared shocks [Proposed]

- $\rho_1(\varepsilon^2) = \alpha(1-\beta^2-\alpha\beta)/(1-\beta^2-2\alpha\beta)$, $\rho_k = \rho_1(\alpha+\beta)^{k-1}$; S&P 500 GARCH-N against the sample autocorrelations. Model: A1.

In [ ]:
# Your code here


## Part B — computation and inference

### B1. GARCH-t on the S&P 500 with diagnostics [Solved]

- ARCH tests on $r_t^2$; GARCH(1,1)-t with robust and classical standard errors; $Q(10)$ on $\hat z_t$ and $\hat z_t^2$, LM(5), sign-bias tests.
- Interpretation: which diagnostics fail and what do they suggest?

In [13]:
# Solution
r = rets['sp500']
print('Q(10) on r^2:', np.round(ljung_box(r ** 2, 10), 4), ' LM(5):', np.round(arch_lm(r - r.mean(), 5), 4))
f = FITS[('sp500', 'GARCH', 't')]
print(f.res.summary())
print('classical s.e.:', f.res.model.fit(disp='off', cov_type='classic', starting_values=f.res.params.values).std_err.round(4).to_dict())
z = f.z.dropna()
print('Q(10) z:', np.round(ljung_box(z, 10), 3), ' Q(10) z^2:', np.round(ljung_box(z ** 2, 10), 3), ' LM(5):', np.round(arch_lm(z, 5), 3))
{k: round(v, 3) for k, v in sign_bias_test(z, f.eps).items()}

Q(10) on r^2: [6132.8172    0.    ]  LM(5): [1683.8597    0.    ]
                        Constant Mean - GARCH Model Results                         
Dep. Variable:                        sp500   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -9062.47
Distribution:      Standardized Student's t   AIC:                           18134.9
Method:                  Maximum Likelihood   BIC:                           18169.0
                                              No. Observations:                 6714
Date:                      Wed, Sep 30 2026   Df Residuals:                     6713
Time:                              17:36:19   Df Model:                            1
                                 Mean Model                                 
                 coef    std err          t      P>|t|      95.0% Conf. Int.
---------------

classical s.e.: {'mu': 0.009, 'omega': 0.0029, 'alpha[1]': 0.0101, 'beta[1]': 0.0096, 'nu': 0.4777}


Q(10) z: [2.1858e+01 1.6000e-02]  Q(10) z^2: [13.61   0.192]  LM(5): [5.799 0.326]


{'sign_t': np.float64(3.564),
 'neg_size_t': np.float64(1.129),
 'pos_size_t': np.float64(-2.613),
 'joint': np.float64(40.665),
 'joint_p': np.float64(0.0)}

### B2. GJR vs GARCH on the S&P 500 [Solved]

- $\hat\gamma$ with its robust $t$; LR test of $\gamma = 0$; AIC and BIC.
- Interpretation: is the asymmetry economically large?

In [14]:
# Solution
lr_tests()
show('GJR vs GARCH, S&P 500', OUT['lr']['sp500'])

GJR vs GARCH, S&P 500
  Log-likelihood, GARCH: -9062.4733
  Log-likelihood, GJR: -8944.6470
  AIC, GARCH: 18134.9466
  AIC, GJR: 17901.2940
  BIC, GARCH: 18169.0063
  BIC, GJR: 17942.1657
  LR statistic: 235.6525
  LR p-value, chi2(1): 0.0000
  gamma: 0.2052
  Robust t of gamma: 10.0277
  alpha: 0.0000
  beta: 0.8809


### B3. GJR vs GARCH on the BET [Proposed]

- Same as B2 for the BET. Model: B2.
- Interpretation: do the LR test and BIC agree?

In [ ]:
# Your code here


### B4. Which innovations for Bitcoin? [Proposed]

- GARCH(1,1) with Normal, Student-t, skewed-t and GED innovations; BIC; LR test of $\lambda = 0$; LR of Normal against Student-t with the $\frac12\chi^2_0 + \frac12\chi^2_1$ law. Model: B1, B2.

In [ ]:
# Your code here


### B5. Forecast comparison on the S&P 500 [Solved]

- 2016–2026, yearly re-estimation; QLIKE, Diebold–Mariano against GARCH-t, Mincer–Zarnowitz; Proxy $r_t^2$, ignoring the mean ($\hat\mu^2 \approx 0.006$ vs an average variance of 1.47).
- Interpretation: which models are significantly worse? Why is $R^2$ low?

In [17]:
# Solution
forecast_comparison('sp500')
pd.DataFrame(OUT['fc_sp500']['models']).T.round(4)

,qlike,dm_diff,dm_t,dm_p,mz_a,mz_b,mz_b_se,mz_r2,mz_wald_p
GARCH-N,0.7510,-0.0060,-1.4733,0.1407,0.0250,0.9734,0.2728,0.2498,0.9951
GARCH-t,0.7570,0.0000,NaN,NaN,0.0844,0.8799,0.2501,0.2380,0.8042
GJR-t,0.7319,-0.0250,-1.2177,0.2233,0.1298,0.8352,0.2168,0.2764,0.6802
EGARCH-t,0.7454,-0.0115,-0.3584,0.7200,-0.2332,1.2381,0.3059,0.2933,0.7130
EWMA,0.8149,0.0579,3.4770,0.0005,0.1413,0.8863,0.2723,0.1718,0.6851
Hist-20,0.8888,0.1319,4.1020,0.0000,0.3997,0.6845,0.2243,0.1519,0.0277


### B5 Extended. Giacomini–White and the Model Confidence Set [Solved]

- Giacomini–White with a rolling 1000-day window, refit each January; the 90% Model Confidence Set on the expanding-window QLIKE losses ($T_R$, stationary bootstrap, mean block 10, 10,000 draws); S&P 500 and BET.

In [18]:
# Solution
GWCOLS = {'mean_diff': 'mean loss difference', 'uncond_t': 'unconditional t', 'gw': 'GW statistic', 'gw_p': 'GW p-value'}
for k in ['sp500', 'bet']:
    gw_mcs(k)
    print(k, 'against GARCH-t')
    print(pd.DataFrame(INF[f'gw_{k}']['tests']).T.rename(columns=GWCOLS).round(3))
    print('90% Model Confidence Set:', INF[f'gw_{k}']['mcs_in'], ' MCS p-values:', INF[f'gw_{k}']['mcs_p'])

sp500 against GARCH-t
          unconditional t  GW statistic  GW p-value  mean loss difference
GARCH-N             1.868         5.850       0.054                 0.008
GJR-t              -0.177         2.490       0.288                -0.005
EGARCH-t            0.075         3.213       0.201                 0.003
EWMA                3.200        11.399       0.003                 0.072
Hist-20             3.873        16.412       0.000                 0.146
90% Model Confidence Set: ['EGARCH-t', 'GARCH-N', 'GARCH-t', 'GJR-t']  MCS p-values: {'Hist-20': 0.001, 'EWMA': 0.0084, 'GARCH-t': 0.3532, 'GARCH-N': 0.5856, 'EGARCH-t': 0.5856, 'GJR-t': 1.0}


bet against GARCH-t
          unconditional t  GW statistic  GW p-value  mean loss difference
GARCH-N             0.948         0.897       0.639                 0.012
GJR-t              -1.708         3.364       0.186                -0.022
EGARCH-t           -1.231         2.587       0.274                -0.021
EWMA                3.252        11.132       0.004                 0.100
Hist-20             3.653        14.214       0.001                 0.147
90% Model Confidence Set: ['EGARCH-t', 'EWMA', 'GARCH-N', 'GARCH-t', 'GJR-t']  MCS p-values: {'Hist-20': 0.0263, 'GARCH-t': 0.1082, 'EWMA': 0.1082, 'GARCH-N': 0.6585, 'EGARCH-t': 0.6585, 'GJR-t': 1.0}


### B6. Forecast comparison on the BET [Proposed]

- Repeat B5 for the BET: mean QLIKE, DM $t$ against GARCH-t, cumulative $\sum d_t$; the Mincer–Zarnowitz regression for GARCH-t with the calibration chart; S&P 500 and BET side by side. Model: B5.

In [ ]:
# Your code here


### B7. Is α+β = 1? Bootstrap and Boundary Test [Solved]

- 999 paths simulated from the fitted S&P 500 GARCH(1,1)-t, re-estimated; percentile intervals for $\alpha+\beta$ and the half-life.
- Algorithm 2, test of IGARCH: 999 paths simulated under $H_0$ from the fitted IGARCH(1,1)-t; bootstrap $p$-values of LR and of $\hat\alpha+\hat\beta$.
- Interpretation: why is a percentile interval truncated at 1 not a test of IGARCH?

In [20]:
# Solution
boots = {'sp500': param_bootstrap('sp500')[0]}
nulls = {'sp500': igarch_test('sp500')}
show('Parametric bootstrap, S&P 500', OUT['boot_sp500'])
show('IGARCH test under H0, S&P 500', OUT['igarch_sp500'])

Parametric bootstrap, S&P 500
  Bootstrap paths B: 999
  Estimated persistence p: 0.9947
  Half-life (days): 130.7335
  2.5% percentile of p*: 0.9833
  97.5% percentile of p*: 1.0000
  Standard deviation of p*: 0.0047
  Mean of p*: 0.9935
  Half-life at the lower end of p: 41.0713
  Median half-life (days): 114.8431
  97.5% percentile of the half-life: inf
  Share of draws with an infinite half-life: 0.0891
  Share of draws with p* >= 0.999: 0.1301
  Robust s.e. of alpha + beta: 0.0047
IGARCH test under H0, S&P 500
  Bootstrap paths B: 999
  Persistence p = alpha + beta: 0.9947
  alpha under H0 (IGARCH-t): 0.1272
  omega under H0: 0.0142
  nu under H0: 6.0336
  LR statistic: 1.1080
  p-value, naive chi2(1): 0.2925
  p-value, mixture 1/2 chi2(0) + 1/2 chi2(1): 0.1463
  Bootstrap p-value of alpha + beta: 0.1670
  Bootstrap p-value of LR: 0.1750
  Share of re-estimates with alpha + beta >= 0.999: 0.5205
  95% quantile of LR*: 3.3595
  5% quantile of p* under H0: 0.9908


### B8. Bootstrap for the BET [Proposed]

- Repeat B7 for the BET, including the test of IGARCH under $H_0$. Model: B7.

In [ ]:
# Your code here


## Part C — open questions and AI critique

### C1. Does US news drive BET volatility? [Proposed]

- Align BET and S&P 500 prices on common days, then compute returns; the US shock for day $t$ is the S&P 500 return of the previous common day.
- $r_t = \mu + \phi u_{t-1} + \varepsilon_t$, $\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2 + \delta u_{t-1}^2$, Student-t; LR test of $\delta = 0$.

In [ ]:
# Your code here


### C2. Critique an AI answer [Proposed]

- The prompt, the AI's code and its output are on the C2 slide of Seminar 5. Model: A3, B1.
- Find the errors, write the corrected code in the cell below and report the long-run volatility and the half-life; log each error in `AI_ERRORS.md`.

In [ ]:
# Your corrected code for C2


### C3. Quasi score-driven GARCH on US stocks [Proposed]

- Project; the tasks are on the C3 slides of Seminar 5. Starter code: the Quantlet `MFM_ch5_quasi_score_driven`.

In [ ]:
# Your code for C3
